# importing all libraries

In [1]:
import os
import re
import sys
import nltk
import itertools
import numpy as np 
import pandas as pd 
import seaborn as sns
from sklearn import tree
from sklearn.svm import SVC
from joblib import dump, load
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from tensorflow.keras.models import Sequential,Model
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from sklearn.ensemble import AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from tensorflow.keras.layers import Dense,LSTM, SpatialDropout1D, Embedding

# Reading the Dataset

In [2]:
df = pd.read_csv(r'UpdatedResumeDataSet.csv')
df

,ID,Resume_str,Resume_html,Category
0,1,"Experienced software engineer with Python, Jav...","<p>Experienced software engineer with Python, ...",Information-Technology
1,2,"HR professional with recruitment experience, t...",<p>HR professional with recruitment experience...,HR
2,3,"Data scientist skilled in Python, R, SQL, and ...","<p>Data scientist skilled in Python, R, SQL, a...",Information-Technology
3,4,"Marketing manager with digital marketing, SEO,...","<p>Marketing manager with digital marketing, S...",Business-Development
4,5,"Teacher with 10 years experience in education,...",<p>Teacher with 10 years experience in educati...,Teacher
5,6,"Lawyer specializing in corporate law, contract...","<p>Lawyer specializing in corporate law, contr...",Advocate
6,7,"Business development executive with sales, neg...","<p>Business development executive with sales, ...",Business-Development
7,8,Healthcare professional with nursing experienc...,<p>Healthcare professional with nursing experi...,Healthcare
8,9,"Fitness trainer with personal training, nutrit...","<p>Fitness trainer with personal training, nut...",Fitness
9,10,"BPO specialist with customer service, call cen...","<p>BPO specialist with customer service, call ...",BPO


# List of all Categories

In [4]:
for i in range(len(df['Category'].unique())):
    print(df['Category'].unique()[i])

Information-Technology
HR
Business-Development
Teacher
Advocate
Healthcare
Fitness
BPO


# Visualizing most commonly used words in each type of Resumes

In [ ]:
a=[ 'Accent_r', 'Blues', 'Blues_r', 'BrBG', 'viridis', 'viridis_r', 'vlag', 'vlag_r', 'winter', 'winter_r','BrBG_r', 'BuGn', 'BuGn_r', 'afmhot', 'afmhot_r', 'autumn', 'autumn_r', 'binary', 'binary_r', 'bone', 'bone_r', 'brg', 'brg_r', 'bwr', 'crest_r']
for label, cmap in zip(df['Category'].unique(), a):
    text = df.query("Category == @label")["Resume"].str.cat(sep=" ")
    plt.figure(figsize=(10, 6))
    wc = WordCloud(width=1000, height=600, background_color="#f8f8f8", colormap=cmap)
    wc.generate_from_text(text)
    plt.imshow(wc)
    plt.axis("off")
    plt.title(f"Words Commonly Used in ${label}$ Resumes", size=20)
    plt.show()

# Pre Processing

# Checking for missing data

In [5]:
print(df.isnull().sum())

ID             0
Resume_str     0
Resume_html    0
Category       0
dtype: int64


# Converting the data into lower case and removing words with small lengths

In [3]:
df['Resume_str'] = df['Resume_str'].apply(lambda x:x.lower())
for i in range(len(df)):
    lw=[]
    for j in df['Resume_str'][i].split():
        if len(j)>=3:
            lw.append(j)
    df['Resume_str'][i]=" ".join(lw)

C:\Users\KARTIK S RATHOD\AppData\Local\Temp\ipykernel_34308\550789070.py:7: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  df['Resume_str'][i]=" ".join(lw)


# removing punctuations

In [4]:
ps = list(";?.:!,")
df['Resume_str'] = df['Resume_str']

for p in ps:   
    df['Resume_str'] = df['Resume_str'].str.replace(p, '')

# Removing '\n' and '\t', extra spaces, quoting text and progressive pronouns

In [6]:
df['Resume_str'] = df['Resume_str'].str.replace("    ", " ")
df['Resume_str'] = df['Resume_str'].str.replace('"', '')
df['Resume_str'] = df['Resume_str'].apply(lambda x: x.replace('\t', ' '))
df['Resume_str'] = df['Resume_str'].str.replace("'s", "")
df['Resume_str'] = df['Resume_str'].apply(lambda x: x.replace('\n', ' '))

# Applying Lemmatization

In [7]:
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to C:\Users\KARTIK S
[nltk_data]     RATHOD\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\KARTIK S
[nltk_data]     RATHOD\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\KARTIK S
[nltk_data]     RATHOD\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [8]:
wl = WordNetLemmatizer()
nr = len(df)
lis = []
for r in range(0, nr):
    ll = []
    t = df.loc[r]['Resume_str']
    tw = str(t).split(" ")
    for w in tw:
        ll.append(wl.lemmatize(w, pos="v"))
    lt = " ".join(ll)
    lis.append(lt)

In [9]:
df['Resume_str'] = lis

# Removing Stop-words

In [10]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\KARTIK S
[nltk_data]     RATHOD\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [11]:
sw = list(stopwords.words('english'))
for s in sw:
    rs = r"\b" + s + r"\b"
    df['Resume_str'] = df['Resume_str'].str.replace(rs, '')

# Visualizing most commonly used words in Resumes after applying NLP techniques 

In [ ]:
a=[ 'Accent_r', 'Blues', 'Blues_r', 'BrBG', 'viridis', 'viridis_r', 'vlag', 'vlag_r', 'winter', 'winter_r','BrBG_r', 'BuGn', 'BuGn_r', 'afmhot', 'afmhot_r', 'autumn', 'autumn_r', 'binary', 'binary_r', 'bone', 'bone_r', 'brg', 'brg_r', 'bwr', 'crest_r','Accent_r', 'Blues', 'Blues_r', 'BrBG', 'viridis', 'viridis_r', 'vlag', 'vlag_r', 'winter', 'winter_r','BrBG_r', 'BuGn', 'BuGn_r', 'afmhot', 'afmhot_r', 'autumn', 'autumn_r', 'binary', 'binary_r', 'bone', 'bone_r', 'brg', 'brg_r', 'bwr']
for label, cmap in zip(df['Category'].unique(), a):
    text = df.query("Category == @label")["Resume_str"].str.cat(sep=" ")
    plt.figure(figsize=(10, 6))
    wc = WordCloud(width=1000, height=600, background_color="#f8f8f8", colormap=cmap)
    wc.generate_from_text(text)
    plt.imshow(wc)

In [ ]:
df.iloc[1,1]

# Data Preparation for Training and Testing

# Encoding Labels

In [12]:
c = LabelEncoder()
df['Category'] = c.fit_transform(df['Category'])
le_name_mapping = dict(zip( c.transform(c.classes_),c.classes_))
print(le_name_mapping)

{np.int64(0): 'Advocate', np.int64(1): 'BPO', np.int64(2): 'Business-Development', np.int64(3): 'Fitness', np.int64(4): 'HR', np.int64(5): 'Healthcare', np.int64(6): 'Information-Technology', np.int64(7): 'Teacher'}


# Using TFIDF approach for converting the content in Resumes into vector form

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
cv = TfidfVectorizer(max_features=20000)
X = cv.fit_transform(df['Resume_str'])
y = df['Category']

In [14]:
a = cv.get_feature_names_out()

In [26]:
import pickle
filename = 'cv.pickle'
pickle.dump(cv, open(filename, 'wb'))

In [16]:
X.shape,y.shape

((10, 75), (10,))

In [18]:
X_res, y_res = X, df['Category']

In [19]:
X_res.shape,y_res.shape

((10, 75), (10,))

# Splitting the Data using Stratified split

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size = 0.25, random_state = 42)

In [22]:
X_train.shape,y_train.shape

((7, 75), (7,))

In [23]:
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                              cmap=plt.cm.Greens):
    plt.figure(figsize=(50, 20), dpi=130)
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, cm[i, j],
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

In [ ]:
print(X_test[0].shape)

# Using KNeighbors Classifier as the Model and printing evaluating it using confusion matrix

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
model1=KNeighborsClassifier()
clf1=GridSearchCV(model1,{'n_neighbors':[1,2,3,4,5,6,7,8,9,10]})
clf1.fit(X_res,y_res)
clf1.cv_results_

In [ ]:
pd1=pd.DataFrame(clf1.cv_results_)
pd1.to_csv('Knn.csv')

In [ ]:
clf1 = KNeighborsClassifier(n_neighbors=1)
clf1 = clf1.fit(X_train, y_train)
yp = clf1.predict(X_test)
acc = accuracy_score(y_test, yp)
print("accuracy is: ",acc)


In [ ]:
yc = clf1.predict(X_res)
CM = confusion_matrix(y_res, yc)
plot_confusion_matrix(CM, classes = range(48),cmap=plt.cm.Blues)
#dump(clf1, 'knei.joblib') 

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
s1 = precision_recall_fscore_support(y_res, yc,average='weighted')
s1

# Using Decision tree as the Model and printing evaluating it using confusion matrix

In [ ]:
from sklearn.tree import DecisionTreeClassifier
model2=DecisionTreeClassifier()
clf2=GridSearchCV(model2,{'max_depth':[10,20,30,40,50,60,70,80,90,100]},cv=5)
clf2.fit(X_res,y_res)
clf2.cv_results_

In [ ]:
pd2=pd.DataFrame(clf2.cv_results_)
pd2.to_csv('DT.csv')

In [ ]:
clf2 = tree.DecisionTreeClassifier(max_depth=100)
clf2 = clf2.fit(X_train, y_train)
yp = clf2.predict(X_test)
acc = accuracy_score(y_test, yp)
print("accuracy is: ",acc)
CM = confusion_matrix(y_test, yp)
plot_confusion_matrix(CM, classes = range(48))
dump(clf2, 'DT.joblib') 

In [ ]:
yp2 = clf2.predict(X_res)

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
s2 = precision_recall_fscore_support(y_res, yp2,average='weighted')
s2

In [ ]:
from sklearn.ensemble import RandomForestClassifier
model3=RandomForestClassifier()
clf3=GridSearchCV(model3,{'n_estimators':[10,50,100,300,500]},cv=5)
clf3.fit(X_res,y_res)

In [ ]:
clf3.best_estimator_

In [ ]:
pd3=pd.DataFrame(clf3.cv_results_)
pd3.to_csv('RF.csv')

In [24]:
clf4=RandomForestClassifier(n_estimators = 500)
clf4 = clf4.fit(X_train, y_train)
yp = clf4.predict(X_test)
acc = accuracy_score(y_test, yp)
print("accuracy is: ",acc)
dump(clf4, 'RF.joblib') 

accuracy is:  0.0


['RF.joblib']

In [ ]:
yp4 = clf4.predict(X_res)
CM = confusion_matrix(y_res, yp4)
plot_confusion_matrix(CM, classes = range(48))

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
s3 = precision_recall_fscore_support(y_res, yp4,average='weighted')
s3

In [ ]:
from sklearn.svm import SVC
model4=SVC()
clf4 = GridSearchCV(model4,{'C':[0.01,0.1,0.5,1],'kernel':['linear','poly','rbf','sigmoid']})
clf4.fit(X_res,y_res)
clf4.cv_results_

In [ ]:
pd4=pd.DataFrame(clf4.cv_results_)
pd4.to_csv('SVC.csv')

In [ ]:
clf4.best_params_

In [ ]:

clf3=SVC(C=1, kernel= 'rbf')
clf3 = clf3.fit(X_train, y_train)
yp = clf3.predict(X_test)
acc = accuracy_score(y_test, yp)
print("accuracy is: ",acc)
dump(clf3, 'SVC.joblib') 



In [ ]:
yp3 = clf3.predict(X_res)
CM = confusion_matrix(y_res, yp3)
plot_confusion_matrix(CM, classes = range(48))

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
s4 = precision_recall_fscore_support(y_res, yp3,average='weighted')
s4

In [ ]:
import xgboost as xgb
xgb_classifier = xgb.XGBClassifier()
xgb_classifier.fit(X_train,y_train)
predictions = xgb_classifier.predict(X_test)
print("Accuracy of Model::",accuracy_score(y_test,predictions))